In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""  # 必須第一行就跑

import torch
print("GPU 可用:", torch.cuda.is_available())  # 應該印出 False

GPU 可用: False


In [5]:
!pip install obspy --quiet
print("安裝完成")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 87.1 MB/s eta 0:00:00
安裝完成


In [1]:
# 先裝好所有需要的套件，版本都固定
!pip install torch==2.4.0 torchvision==0.19.0 --quiet
!pip install transformers==4.44.0 peft==0.12.0 --quiet
!pip install datasets evaluate accelerate --quiet
!pip install obspy --quiet

import sys
# 禁用 torchvision，避免 VideoReader 錯誤
if 'torchvision' in sys.modules:
    del sys.modules['torchvision']
sys.modules['torchvision'] = None

import torch
print("PyTorch:", torch.__version__)
print("GPU 可用:", torch.cuda.is_available())

PyTorch: 2.4.0+cu121
GPU 可用: True


In [2]:
from huggingface_hub import whoami
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

try:
    user_info = whoami(token=hf_token)
    print(f"✅ 登入成功！帳號: {user_info['name']}")
except Exception as e:
    print(f"❌ 登入失敗: {e}")

✅ 登入成功！帳號: huihui0206


In [6]:
#製作資料集的程式碼
import time
import numpy as np
import matplotlib.pyplot as plt
from obspy.clients.fdsn import Client
from obspy import UTCDateTime, Stream
from obspy.taup import TauPyModel
from obspy.geodetics import locations2degrees
from IPython.display import clear_output, display
from datasets import Dataset, DatasetDict

class SeismicDatasetBuilder:
    def __init__(self, client_name="EARTHSCOPE", event_client_name="USGS", model_name="iasp91"):
        """初始化地震資料抓取器"""
        self.client = Client(client_name)           # 波形與測站資料
        self.event_client = Client(event_client_name)  # 地震目錄（USGS 有 event service）
        self.model = TauPyModel(model=model_name)
        self.raw_streams = Stream()
        self.hf_data_dict = {"text": [], "label": []}

    def fetch_catalog(self, start_time, end_time, min_mag=7.0):
        """1. 獲取地震目錄"""
        print(f"正在檢索 {start_time} 附近的地震事件...")
        catalog = self.event_client.get_events(starttime=UTCDateTime(start_time),
                                              endtime=UTCDateTime(end_time),
                                              minmagnitude=min_mag)
        if not catalog:
            print("找不到符合條件的地震。")
            return None

        event = catalog[0]
        origin = event.origins[0]
        info = {
            "time": origin.time,
            "lat": origin.latitude,
            "lon": origin.longitude,
            "depth_km": origin.depth / 1000.0
        }
        print(f"找到事件: {info['time']} | 座標: ({info['lat']}, {info['lon']}) | 深度: {info['depth_km']}km")
        return info

    def get_candidate_stations(self, event_info, max_radius=30):
        """2. 獲取候選測站"""
        print(f"正在檢索震央距 {max_radius} 度內的 BHZ 測站...")
        inventory = self.client.get_stations(
            latitude=event_info['lat'], longitude=event_info['lon'],
            maxradius=max_radius, channel="BHZ",
            starttime=event_info['time'], endtime=event_info['time'] + 3600,
            level="station"
        )

        stations = []
        for network in inventory:
            for sta in network:
                dist = locations2degrees(event_info['lat'], event_info['lon'], sta.latitude, sta.longitude)
                stations.append({'net': network.code, 'sta': sta.code, 'dist': dist})

        return sorted(stations, key=lambda x: x['dist'])

    def download_and_filter(self, event_info, station_list, sampling_rate=20.0, duration=100):
        """3. 自動品質篩選 (20Hz, 無中斷)"""
        self.raw_streams = Stream()
        print(f"開始掃描 {len(station_list)} 個測站的數據...")

        for i, s in enumerate(station_list):
            arrivals = self.model.get_travel_times(
                source_depth_in_km=event_info['depth_km'],
                distance_in_degree=s['dist'],
                phase_list=["P", "Pn", "Pg", "p"]
            )
            if not arrivals: continue

            p_arrival = event_info['time'] + arrivals[0].time

            try:
                # 抓取緩衝範圍 (P-20s ~ P+100s)
                st = self.client.get_waveforms(
                    network=s['net'], station=s['sta'], location="*",
                    channel="BHZ", starttime=p_arrival - 20, endtime=p_arrival + 100
                )

                for tr in st:
                    # 品質檢查
                    if abs(tr.stats.sampling_rate - sampling_rate) > 0.001: continue
                    if len(st.select(id=tr.id)) > 1: continue # 有間斷
                    if tr.stats.npts < (duration + 20) * sampling_rate - 50: continue # 長度不足

                    tr.stats.p_arrival = p_arrival
                    self.raw_streams.append(tr)
            except:
                pass

            if i % 10 == 0:
                print(f"進度: {i}/{len(station_list)} 測站已掃描...", end="\r")

        print(f"\n篩選完成，共 {len(self.raw_streams)} 條候選波形。")

    def manual_review(self, target_points=2000):
        """4. 人工審核與數據裁切"""
        if not self.raw_streams:
            print("沒有波形可供審核。")
            return

        for tr in self.raw_streams:
            clear_output(wait=True)
            print(f"🔍 審核中: {tr.id} (已保留: {self.hf_data_dict['label'].count(1)})")
            print("👉 y: 保留 (Label 1) | n: 捨棄 (Label 0) | s: 停止審核並產出")

            fig = tr.plot(handle=True, title=f"{tr.id} - Verify P-wave")
            display(fig)
            time.sleep(0.2)

            choice = input("輸入 (y/n/s): ").strip().lower()
            plt.close(fig)

            if choice == 's': break

            # 裁切 (P-10s ~ P+90s)
            p_t = tr.stats.p_arrival
            tr_cut = tr.slice(starttime=p_t - 10, endtime=p_t + 90)

            # 數據對齊 (Padding or Truncate)
            data = tr_cut.data[:target_points].tolist()
            if len(data) < target_points:
                data.extend([data[-1]] * (target_points - len(data)))

            self.hf_data_dict["text"].append(data)
            self.hf_data_dict["label"].append(1 if choice == 'y' else 0)

    def build_dataset(self, test_size=0.2):
        """5. 轉換為 DatasetDict"""
        full_ds = Dataset.from_dict(self.hf_data_dict)
        num_samples = len(full_ds)

        if num_samples >= 5:
            train_test = full_ds.train_test_split(test_size=test_size, seed=42)
            val_test = train_test['test'].train_test_split(test_size=0.5, seed=42)
            ds = DatasetDict({
                'train': train_test['train'],
                'validation': val_test['train'],
                'test': val_test['test']
            })
        else:
            ds = DatasetDict({'train': full_ds})
            print("樣本數過少，僅建立 train 拆分。")

        print("=== 🎉 Dataset 製作完成 ===")
        return ds

In [7]:
# 處理資料集的程式碼
from datasets import load_dataset, concatenate_datasets, DatasetDict
from google.colab import userdata
import logging

# 設定日誌，方便追蹤進度
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

class HFDatasetManager:
    def __init__(self, repo_id, token_name='HF_TOKEN'):
        """
        初始化管理員
        :param repo_id: Hugging Face 的 Repository ID
        :param token_name: 儲存在 Colab Secret 中的 Token 名稱
        """
        self.repo_id = repo_id
        try:
            self.token = userdata.get(token_name)
        except Exception:
            self.token = None
            logging.warning(f"未能在 userdata 中找到 {token_name}，將以公開模式嘗試操作。")

        self.dataset = None

    def download(self):
        """從 Hub 下載資料集"""
        logging.info(f"正在從 {self.repo_id} 載入資料集...")
        try:
            self.dataset = load_dataset(self.repo_id, token=self.token)
            logging.info("=== 資料集載入成功 ===")
            print(self.dataset)
            return self.dataset
        except Exception as e:
            logging.error(f"下載失敗: {e}")
            return None

    def merge_with(self, new_ds):
        """
        將目前的資料集與傳入的新資料集合併
        :param new_ds: 另一個 DatasetDict 或 Dataset 物件
        """
        if self.dataset is None:
            logging.warning("目前沒有已載入的資料集，將直接使用傳入的資料集。")
            self.dataset = new_ds
            return self.dataset

        logging.info("準備合併資料集...")
        merged_splits = {}
        # 找出兩個資料集中所有的 split (例如 'train', 'test')
        all_splits = set(self.dataset.keys()).union(set(new_ds.keys()))

        for split in all_splits:
            to_concat = []
            if split in self.dataset:
                to_concat.append(self.dataset[split])
            if split in new_ds:
                to_concat.append(new_ds[split])

            if len(to_concat) > 1:
                merged_splits[split] = concatenate_datasets(to_concat)
            else:
                merged_splits[split] = to_concat[0]

        self.dataset = DatasetDict(merged_splits)
        logging.info("=== 🎉 合併完成 ===")
        print(self.dataset)
        return self.dataset

    def upload(self, private=False):
        """將目前的資料集推送到 Hugging Face Hub"""
        if self.dataset is None:
            logging.error("沒有資料集可以上傳！")
            return

        logging.info(f"準備將資料集推送到 {self.repo_id}...")
        try:
            self.dataset.push_to_hub(self.repo_id, token=self.token, private=private)
            logging.info(f"✅ 成功！網址: https://huggingface.co/datasets/{self.repo_id}")
        except Exception as e:
            logging.error(f"上傳失敗: {e}")

    def preview(self, split='train', idx=0):
        """預覽特定資料"""
        if self.dataset and split in self.dataset:
            sample = self.dataset[split][idx]
            print(f"\n[預覽] {split} 集第 {idx} 筆資料:")
            for key, value in sample.items():
                content = value[:5] if isinstance(value, list) else value
                print(f"- {key}: {content} (類型: {type(value)})")
        else:
            print("無法預覽，資料集尚未載入或 Split 不存在。")

In [ ]:
# 在這裡填入要抓取的地震資料
# 範例：
# builder = SeismicDatasetBuilder()
# event = builder.fetch_catalog(
#     start_time="2024-04-03T00:00:00",
#     end_time="2024-04-03T23:59:00",
#     min_mag=5.0
# )
# if event:
#     stations = builder.get_candidate_stations(event, max_radius=30)
#     builder.download_and_filter(event, stations)
#     builder.manual_review()
#     from datasets import Dataset, DatasetDict
#     full_ds = Dataset.from_dict(builder.hf_data_dict)
#     my_seismic_ds = DatasetDict({'train': full_ds})
#     print(my_seismic_ds)

In [5]:
from datasets import DatasetDict, Dataset
from collections import Counter

MY_REPO = "huihui0206/huihui"
manager = HFDatasetManager(MY_REPO)
old_ds = manager.download()

def to_float(ds_split):
    texts = []
    labels = []
    for row in ds_split:
        texts.append([float(x) for x in row['text']])
        labels.append(int(row['label']))
    return Dataset.from_dict({"text": texts, "label": labels})

merged_train = Dataset.from_dict({
    "text": [row['text'] for row in to_float(old_ds['train'])] +
            [row['text'] for row in to_float(my_seismic_ds['train'])],
    "label": [row['label'] for row in to_float(old_ds['train'])] +
             [row['label'] for row in to_float(my_seismic_ds['train'])]
})

fixed_ds = DatasetDict({
    'train': merged_train,
    'validation': to_float(old_ds['validation']),
    'test': to_float(old_ds['test'])
})

manager.dataset = fixed_ds
manager.upload()

print(manager.dataset)
print(Counter(manager.dataset['train']['label']))

README.md:   0%|          | 0.00/520 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/20.3M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/117k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/167k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1568 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/12 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 1568
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 10
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 12
    })
})


NameError: name 'my_seismic_ds' is not defined

In [8]:
from datasets import DatasetDict, Dataset
from collections import Counter

MY_REPO = "huihui0206/huihui"
manager = HFDatasetManager(MY_REPO)
old_ds = manager.download()

# 統一轉成 float，確保格式一致
def to_float(ds_split):
    texts = []
    labels = []
    for row in ds_split:
        texts.append([float(x) for x in row['text']])
        labels.append(int(row['label']))
    return Dataset.from_dict({"text": texts, "label": labels})

ds = DatasetDict({
    'train': to_float(old_ds['train']),
    'validation': to_float(old_ds['validation']),
    'test': to_float(old_ds['test'])
})

print(ds)
print(Counter(ds['train']['label']))

README.md:   0%|          | 0.00/520 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/20.3M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/117k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/167k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1568 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/12 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 1568
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 10
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 12
    })
})
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 1568
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 10
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 12
    })
})
Counter({0: 845, 1: 723})


In [10]:
!pip install evaluate --quiet
print("安裝完成")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.6 MB/s eta 0:00:00
安裝完成


In [16]:
from datasets import DatasetDict, Dataset
from collections import Counter

# 重新從 train 切出更多 validation 和 test
full_train = ds['train']

# 切出 20% 當測試
train_test = full_train.train_test_split(test_size=0.2, seed=42)
# 再切一半當 validation
val_test = train_test['test'].train_test_split(test_size=0.5, seed=42)

ds = DatasetDict({
    'train': train_test['train'],
    'validation': val_test['train'],
    'test': val_test['test']
})

print(ds)
print(Counter(ds['train']['label']))

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 1254
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 157
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 157
    })
})
Counter({0: 680, 1: 574})


In [17]:
import numpy as np
import os
import torch
import torch.nn as nn
import collections
from torch.utils.data import Dataset as TorchDataset, DataLoader
from torch.optim import AdamW
os.environ["WANDB_DISABLED"] = "true"
os.environ["CUDA_VISIBLE_DEVICES"] = ""

print("🔄 正在準備資料...")

class SeismicTorchDataset(TorchDataset):
    def __init__(self, hf_dataset):
        self.data = hf_dataset
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        row = self.data[idx]
        waveform = torch.tensor(row['text'], dtype=torch.float32).unsqueeze(0)
        label = torch.tensor(int(row['label']), dtype=torch.long)
        return waveform, label

train_dataset = SeismicTorchDataset(ds['train'])
val_dataset   = SeismicTorchDataset(ds['validation'])
test_dataset  = SeismicTorchDataset(ds['test'])

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=16)
test_loader  = DataLoader(test_dataset,  batch_size=16)

label_counts = collections.Counter(ds['train']['label'])
print(f"Label 分布: {label_counts}")
print(f"訓練集: {len(train_dataset)} 條")

class SeismicCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=7, padding=3), nn.BatchNorm1d(16), nn.ReLU(), nn.MaxPool1d(4),
            nn.Conv1d(16, 32, kernel_size=7, padding=3), nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(4),
            nn.Conv1d(32, 64, kernel_size=7, padding=3), nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(4),
            nn.Conv1d(64, 64, kernel_size=7, padding=3), nn.BatchNorm1d(64), nn.ReLU(), nn.AdaptiveAvgPool1d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, 32), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(32, 2)
        )
    def forward(self, x):
        return self.classifier(self.features(x))

device = torch.device("cpu")
model = SeismicCNN().to(device)
optimizer = AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
criterion = nn.CrossEntropyLoss()

print(f"模型參數量: {sum(p.numel() for p in model.parameters()):,}")
print("🚀 開始訓練（CPU 模式，約 20~40 分鐘）...")

best_val_acc = 0
num_epochs = 30  # 從 10 增加到 30

for epoch in range(num_epochs):
    model.train()
    train_loss, train_correct = 0, 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        train_correct += (out.argmax(1) == y).sum().item()

    model.eval()
    val_correct = 0
    with torch.no_grad():
        for x, y in val_loader:
            out = model(x.to(device))
            val_correct += (out.argmax(1) == y.to(device)).sum().item()

    train_acc = train_correct / len(train_dataset)
    val_acc = val_correct / len(val_dataset)
    print(f"Epoch {epoch+1:2d}/{num_epochs} | Loss: {train_loss/len(train_loader):.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "./best_seismic_model.pt")
        print(f"  ✨ 新最佳模型已儲存！Val Acc: {val_acc:.4f}")

model.load_state_dict(torch.load("./best_seismic_model.pt"))
model.eval()
test_correct = 0
with torch.no_grad():
    for x, y in test_loader:
        out = model(x.to(device))
        test_correct += (out.argmax(1) == y.to(device)).sum().item()

test_acc = test_correct / len(test_dataset)
print(f"\n🎉 訓練完成！")
print(f"✅ 最佳驗證集準確率: {best_val_acc:.4f}")
print(f"✅ 測試集準確率: {test_acc:.4f}")

🔄 正在準備資料...
Label 分布: Counter({0: 680, 1: 574})
訓練集: 1254 條
模型參數量: 49,378
🚀 開始訓練（CPU 模式，約 20~40 分鐘）...
Epoch  1/30 | Loss: 0.5300 | Train Acc: 0.7448 | Val Acc: 0.7962
  ✨ 新最佳模型已儲存！Val Acc: 0.7962
Epoch  2/30 | Loss: 0.3629 | Train Acc: 0.8573 | Val Acc: 0.8981
  ✨ 新最佳模型已儲存！Val Acc: 0.8981
Epoch  3/30 | Loss: 0.3489 | Train Acc: 0.8573 | Val Acc: 0.8089
Epoch  4/30 | Loss: 0.3780 | Train Acc: 0.8413 | Val Acc: 0.8153
Epoch  5/30 | Loss: 0.3524 | Train Acc: 0.8628 | Val Acc: 0.7898
Epoch  6/30 | Loss: 0.3427 | Train Acc: 0.8620 | Val Acc: 0.8535
Epoch  7/30 | Loss: 0.3269 | Train Acc: 0.8756 | Val Acc: 0.5096
Epoch  8/30 | Loss: 0.3415 | Train Acc: 0.8724 | Val Acc: 0.8408
Epoch  9/30 | Loss: 0.3174 | Train Acc: 0.8892 | Val Acc: 0.8535
Epoch 10/30 | Loss: 0.3179 | Train Acc: 0.8844 | Val Acc: 0.9045
  ✨ 新最佳模型已儲存！Val Acc: 0.9045
Epoch 11/30 | Loss: 0.2994 | Train Acc: 0.8995 | Val Acc: 0.5096
Epoch 12/30 | Loss: 0.2982 | Train Acc: 0.8828 | Val Acc: 0.8408
Epoch 13/30 | Loss: 0.2723 | T